In [11]:
import pandas as pd
import numpy as np
import yfinance as yf
from scipy.stats import norm
import os

In [12]:
#Downloading the ticket data
ticker = "HDFCBANK.NS"

if os.path.exists("hdfcbank_data.csv"):
    data = pd.read_csv("hdfcbank_data.csv", index_col=0, parse_dates=True)
    data["Close"] = pd.to_numeric(data["Close"], errors="coerce")
    data = data.dropna()
else:
    data = yf.download("HDFCBANK.NS", period="6mo", interval="1d")
    data.to_csv("hdfcbank_data.csv")

/var/folders/3t/3xjt3f516dx8r11ngt2k1w_m0000gn/T/ipykernel_73713/2724481629.py:5: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  data = pd.read_csv("hdfcbank_data.csv", index_col=0, parse_dates=True)


In [13]:
#Calculating the volatility
data["Log Return"] = np.log(data["Close"] / data["Close"].shift(1))
data = data.dropna()
voldaily = data["Log Return"].std()
vol = voldaily * np.sqrt(252)
print("Daily Volatility:", voldaily)
print("Annualized Historical Volatility:", vol)

Daily Volatility: 0.013974403809367516
Annualized Historical Volatility: 0.22183678319988068


In [14]:
#Defining the BSM functions
def black_scholes(S, K, T, r, sigma, option_type="call"):
    d1 = (np.log(S / K) + (r + 0.5 * sigma ** 2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    if option_type == "call":
        return S * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)
    else:
        return K * np.exp(-r * T) * norm.cdf(-d2) - S * norm.cdf(-d1)

In [ ]:
#The data was collected on 11th April 2026
#Options chains data
SNAPSHOT_DATE = "2026-04-11"
S = 810.30  # HDFCBANK spot on snapshot date
r = 0.053 # Risk-free rate 91 Day T-Bill yield as of snapshot date

rows = [
    ("28-Apr", 16, "ATM Call", 810, 20.30),
    ("28-Apr", 16, "ATM Put",  810, 18.95),
    ("28-Apr", 16, "OTM Call", 870,  3.50),
    ("28-Apr", 16, "OTM Put",  750,  4.35),
    ("26-May", 44, "ATM Call", 810, 31.05),
    ("26-May", 44, "ATM Put",  810, 21.70),
    ("26-May", 44, "OTM Call", 870, 11.15),
    ("26-May", 44, "OTM Put",  750,  9.50),
]

df = pd.DataFrame(rows, columns=["Expiry", "Days", "Option", "Strike", "Market Price"])
print(df)

In [ ]:
bsm_prices = []
for i in range(len(df)):
    K = df.loc[i, "Strike"]
    T = df.loc[i, "Days"] / 365
    opt_type = df.loc[i, "Option"]
    if "Call" in opt_type:
        price = black_scholes(S, K, T, r, vol, "call")
    else:
        price = black_scholes(S, K, T, r, vol, "put")
    bsm_prices.append(price)

df["BSM Price"] = bsm_prices
df["Difference"] = df["Market Price"] - df["BSM Price"]
df["Spot Price (S)"] = S
df["Volatility (vol)"] = vol
df["Risk-Free Rate (r)"] = r

print(f"Stock: HDFCBANK | Spot: ₹{S} | Vol: {round(vol*100, 2)}% | r: {r}")
print(df[["Expiry", "Option", "Strike", "Market Price", "BSM Price", "Difference"]].to_string(index=False))

Stock: HDFCBANK | Spot: ₹810.3 | Vol: 22.18% | r: 0.053
Expiry   Option  Strike  Market Price  BSM Price  Difference
28-Apr ATM Call     810         20.30  16.107727    4.192273
28-Apr  ATM Put     810         18.95  13.928049    5.021951
28-Apr OTM Call     870          3.50   1.192895    2.307105
28-Apr  OTM Put     750          4.35   0.637465    3.712535
26-May ATM Call     810         31.05  27.633021    3.416979
26-May  ATM Put     810         21.70  22.174395   -0.474395
26-May OTM Call     870         11.15   7.213962    3.936038
26-May  OTM Put     750          9.50   4.211897    5.288103


In [ ]:
df.to_excel("HDFCBANK_BSM_Comparison.xlsx", index=False)
print("Saved as HDFCBANK_BSM_Comparison.xlsx")